In [6]:
from pathlib import Path
import pyarrow.parquet as pq
import pandas as pd

BASE = Path("../data/processed/checkpoints")

files = [
    "graph_train_raw.parquet",
    "graph_val_raw.parquet",
    "graph_test_raw.parquet",
    "train_filled.parquet",
    "val_filled.parquet",
    "train_pre_scaling.parquet",
    "val_pre_scaling.parquet",
    "test_pre_scaling.parquet",
    "train_normalized.parquet",
    "val_normalized.parquet",
    "test_normalized.parquet",
]

for name in files:
    path = BASE / name

    print("\n" + "=" * 80)
    print(name)
    print("=" * 80)

    pf = pq.ParquetFile(path)

    print("Rows:", pf.metadata.num_rows)
    print("Columns:", pf.schema_arrow.names)


graph_train_raw.parquet
Rows: 5553096
Columns: ['stay_id', 'hour', 'resp_rate', 'spo2', 'heart_rate', 'sbp', 'temperature', 'lactate', 'creatinine', 'wbc', 'resp_rate_missing', 'spo2_missing', 'heart_rate_missing', 'sbp_missing', 'temperature_missing', 'lactate_missing', 'creatinine_missing', 'wbc_missing']

graph_val_raw.parquet
Rows: 1169538
Columns: ['stay_id', 'hour', 'resp_rate', 'spo2', 'heart_rate', 'sbp', 'temperature', 'lactate', 'creatinine', 'wbc', 'resp_rate_missing', 'spo2_missing', 'heart_rate_missing', 'sbp_missing', 'temperature_missing', 'lactate_missing', 'creatinine_missing', 'wbc_missing']

graph_test_raw.parquet
Rows: 1190145
Columns: ['stay_id', 'hour', 'resp_rate', 'spo2', 'heart_rate', 'sbp', 'temperature', 'lactate', 'creatinine', 'wbc', 'resp_rate_missing', 'spo2_missing', 'heart_rate_missing', 'sbp_missing', 'temperature_missing', 'lactate_missing', 'creatinine_missing', 'wbc_missing']

train_filled.parquet
Rows: 5767242
Columns: ['hour', 'stay_id', 'resp_ra

In [7]:
from pathlib import Path
import pyarrow.parquet as pq
import pandas as pd

BASE = Path("../data/processed/checkpoints")

files = [
    "graph_train_raw.parquet",
    "graph_val_raw.parquet",
    "graph_test_raw.parquet",
    "train_filled.parquet",
    "val_filled.parquet",
    "train_normalized.parquet",
    "val_normalized.parquet",
    "test_normalized.parquet",
]

for name in files:

    path = BASE / name
    pf = pq.ParquetFile(path)
    columns = pf.schema_arrow.names

    print("\n" + "=" * 80)
    print(name)
    print("=" * 80)

    print("Rows:", pf.metadata.num_rows)

    # ID columns
    id_cols = [
        c for c in ["subject_id", "stay_id", "hour", "intime"]
        if c in columns
    ]

    # Possible label columns
    label_cols = [
        c for c in columns
        if any(x in c.lower() for x in [
            "label", "target", "deterior", "outcome", "event"
        ])
    ]

    # Missingness columns
    missing_cols = [
        c for c in columns
        if c.endswith("_missing")
    ]

    selected = list(dict.fromkeys(
        id_cols + label_cols + missing_cols
    ))

    print("ID columns:", id_cols)
    print("Possible label columns:", label_cols)
    print("Missingness columns:", missing_cols)

    if not selected:
        print("Nothing to audit in this file.")
        continue

    df = pd.read_parquet(path, columns=selected)

    # Unique IDs
    for c in ["subject_id", "stay_id"]:
        if c in df.columns:
            print(
                f"Unique {c}:",
                df[c].nunique()
            )

    # Label statistics
    for c in label_cols:
        print(f"\nLABEL: {c}")
        print(df[c].value_counts(dropna=False).sort_index())
        print("Positive rate:", df[c].mean())

    # Missingness statistics
    if missing_cols:
        print("\nMISSINGNESS:")
        for c in missing_cols:
            print(
                f"{c:30s}",
                f"count={df[c].sum():,.0f}",
                f"rate={df[c].mean()*100:.3f}%"
            )


graph_train_raw.parquet
Rows: 5553096
ID columns: ['stay_id', 'hour']
Possible label columns: []
Missingness columns: ['resp_rate_missing', 'spo2_missing', 'heart_rate_missing', 'sbp_missing', 'temperature_missing', 'lactate_missing', 'creatinine_missing', 'wbc_missing']
Unique stay_id: 66107

MISSINGNESS:
resp_rate_missing              count=78,636 rate=1.416%
spo2_missing                   count=110,328 rate=1.987%
heart_rate_missing             count=20,060 rate=0.361%
sbp_missing                    count=2,040,529 rate=36.746%
temperature_missing            count=5,311,536 rate=95.650%
lactate_missing                count=5,357,516 rate=96.478%
creatinine_missing             count=5,157,662 rate=92.879%
wbc_missing                    count=5,206,191 rate=93.753%

graph_val_raw.parquet
Rows: 1169538
ID columns: ['stay_id', 'hour']
Possible label columns: []
Missingness columns: ['resp_rate_missing', 'spo2_missing', 'heart_rate_missing', 'sbp_missing', 'temperature_missing', 'lactat

In [9]:
from pathlib import Path
import pandas as pd

BASE = Path("../data/processed/checkpoints")

files = [
    "graph_train_raw.parquet",
    "graph_val_raw.parquet",
    "graph_test_raw.parquet",
    "train_filled.parquet",
    "val_filled.parquet",
    "train_normalized.parquet",
    "val_normalized.parquet",
    "test_normalized.parquet",
]

for name in files:

    path = BASE / name

    print("\n" + "=" * 70)
    print(name)
    print("=" * 70)

    df = pd.read_parquet(path)

    print("Rows:", len(df))
    print("Unique stays:", df["stay_id"].nunique())

    missing_cols = [
        c for c in df.columns
        if c.endswith("_missing")
    ]

    if missing_cols:
        print("\nMissingness rates:")
        for c in missing_cols:
            print(
                f"{c:30s} "
                f"{df[c].mean()*100:.3f}%"
            )

    print("\nColumns:")
    print(df.columns.tolist())

    del df


graph_train_raw.parquet
Rows: 5553096
Unique stays: 66107

Missingness rates:
resp_rate_missing              1.416%
spo2_missing                   1.987%
heart_rate_missing             0.361%
sbp_missing                    36.746%
temperature_missing            95.650%
lactate_missing                96.478%
creatinine_missing             92.879%
wbc_missing                    93.753%

Columns:
['stay_id', 'hour', 'resp_rate', 'spo2', 'heart_rate', 'sbp', 'temperature', 'lactate', 'creatinine', 'wbc', 'resp_rate_missing', 'spo2_missing', 'heart_rate_missing', 'sbp_missing', 'temperature_missing', 'lactate_missing', 'creatinine_missing', 'wbc_missing']

graph_val_raw.parquet
Rows: 1169538
Unique stays: 14164

Missingness rates:
resp_rate_missing              1.450%
spo2_missing                   2.039%
heart_rate_missing             0.378%
sbp_missing                    35.736%
temperature_missing            95.737%
lactate_missing                96.524%
creatinine_missing             9

In [10]:
from pathlib import Path
import pyarrow.parquet as pq
import pandas as pd

BASE = Path("../data/processed/checkpoints")

for name in [
    "train_normalized.parquet",
    "val_normalized.parquet",
    "test_normalized.parquet"
]:

    path = BASE / name

    df = pd.read_parquet(
        path,
        columns=["stay_id", "hour"]
    )

    df["hour"] = pd.to_datetime(df["hour"])

    # Number of rows for each ICU stay
    lengths = df.groupby("stay_id").size()

    # For W=24 and stride=1:
    windows = (lengths - 24 + 1).clip(lower=0)

    print("\n", name)
    print("Hourly rows:", len(df))
    print("Unique stays:", len(lengths))
    print("24h windows:", int(windows.sum()))
    print("Positive count from training:", 
          {"train_normalized.parquet": 6383,
           "val_normalized.parquet": 1335,
           "test_normalized.parquet": None}.get(name))
    
    del df


 train_normalized.parquet
Hourly rows: 5767242
Unique stays: 66107
24h windows: 4344379
Positive count from training: 6383

 val_normalized.parquet
Hourly rows: 1232929
Unique stays: 14164
24h windows: 927984
Positive count from training: 1335

 test_normalized.parquet
Hourly rows: 1234621
Unique stays: 14167
24h windows: 929999
Positive count from training: None


In [2]:
from pathlib import Path
import pandas as pd
import numpy as np


# ============================================================
# PATHS
# ============================================================

EVENTS_PATH = "../data/processed/composite_events.csv"

SPLITS = {
    "train": {
        "features": "data/processed/checkpoints/train_normalized.parquet",
        "labels": "data/processed/train_labels.parquet",
    },
    "val": {
        "features": "data/processed/checkpoints/val_normalized.parquet",
        "labels": "data/processed/val_labels.parquet",
    },
    "test": {
        "features": "data/processed/checkpoints/test_normalized.parquet",
        "labels": "data/processed/test_labels.parquet",
    },
}


# ============================================================
# LOAD EVENTS
# ============================================================

print("\n" + "=" * 80)
print("COMPOSITE EVENT AUDIT")
print("=" * 80)

events = pd.read_csv(EVENTS_PATH)

event_cols = [
    "death_hour",
    "vasopressor_hour",
    "ventilation_hour",
    "sepsis_hour",
]

for col in event_cols:
    if col not in events.columns:
        events[col] = np.nan

events["deterioration_hour"] = events[event_cols].min(
    axis=1,
    skipna=True
)

events["has_event"] = events["deterioration_hour"].notna()

print(f"\nTotal ICU stays in composite_events : {len(events):,}")
print(
    f"Stays with deterioration event      : "
    f"{events['has_event'].sum():,}"
)
print(
    f"Stays without deterioration event   : "
    f"{(~events['has_event']).sum():,}"
)

print("\nEarliest deterioration hour distribution:")

event_hours = events.loc[
    events["has_event"],
    "deterioration_hour"
]

print(event_hours.describe())

print("\nEvents occurring early:")

for h in [8, 12, 24, 48, 72]:
    count = (event_hours <= h).sum()
    print(
        f"  Event <= {h:>2} hours : "
        f"{count:>8,} "
        f"({100 * count / len(event_hours):.2f}%)"
    )


# ============================================================
# CHECK EACH SPLIT
# ============================================================

for split, paths in SPLITS.items():

    print("\n" + "=" * 80)
    print(f"{split.upper()} SPLIT AUDIT")
    print("=" * 80)

    feature_path = Path(paths["features"])
    label_path = Path(paths["labels"])

    if not feature_path.exists():
        print(f"ERROR: Missing features file:")
        print(f"  {feature_path}")
        continue

    if not label_path.exists():
        print(f"ERROR: Missing labels file:")
        print(f"  {label_path}")
        continue

    # --------------------------------------------------------
    # Load
    # --------------------------------------------------------

    features = pd.read_parquet(feature_path)
    labels = pd.read_parquet(label_path)

    print("\nBasic sizes:")
    print(f"  Feature rows : {len(features):,}")
    print(f"  Feature stays: {features['stay_id'].nunique():,}")
    print(f"  Label rows   : {len(labels):,}")
    print(f"  Label stays  : {labels['stay_id'].nunique():,}")

    # --------------------------------------------------------
    # Label distribution
    # --------------------------------------------------------

    n_pos = int(labels["label"].sum())
    n_total = len(labels)
    n_neg = n_total - n_pos

    print("\nLabel distribution:")
    print(f"  Positive : {n_pos:,}")
    print(f"  Negative : {n_neg:,}")
    print(f"  Total    : {n_total:,}")

    if n_total > 0:
        prevalence = n_pos / n_total
        ratio = n_neg / max(n_pos, 1)

        print(f"  Positive rate : {100 * prevalence:.4f}%")
        print(f"  Imbalance     : {ratio:.1f}:1")

    # --------------------------------------------------------
    # Duplicate stay/hour check
    # --------------------------------------------------------

    duplicates = labels.duplicated(
        subset=["stay_id", "hours"]
    ).sum()

    print("\nDuplicate label timestamps:")
    print(f"  {duplicates:,}")

    # --------------------------------------------------------
    # Check labels against deterioration event
    # --------------------------------------------------------

    check = labels.merge(
        events[
            [
                "stay_id",
                "deterioration_hour"
            ]
        ],
        on="stay_id",
        how="left"
    )

    # Distance from prediction time to event
    check["event_minus_prediction"] = (
        check["deterioration_hour"]
        - check["hours"]
    )

    # --------------------------------------------------------
    # Positive label correctness
    # --------------------------------------------------------

    positive_rows = check[
        check["label"] == 1
    ].copy()

    if len(positive_rows) > 0:

        correct_positive = positive_rows[
            positive_rows["event_minus_prediction"].between(
                6,
                8,
                inclusive="both"
            )
        ]

        incorrect_positive = positive_rows[
            ~positive_rows["event_minus_prediction"].between(
                6,
                8,
                inclusive="both"
            )
        ]

        print("\nPositive-label audit:")
        print(
            f"  Positive rows checked : "
            f"{len(positive_rows):,}"
        )
        print(
            f"  Correct [6,8] window  : "
            f"{len(correct_positive):,}"
        )
        print(
            f"  Outside [6,8] window  : "
            f"{len(incorrect_positive):,}"
        )

        print("\nPositive event-distance distribution:")

        print(
            positive_rows[
                "event_minus_prediction"
            ].value_counts()
            .sort_index()
            .to_string()
        )

    else:
        print("\nNo positive labels found.")

    # --------------------------------------------------------
    # Check negative labels that appear to be inside [6,8]
    # --------------------------------------------------------

    should_be_positive = check[
        check["event_minus_prediction"].between(
            6,
            8,
            inclusive="both"
        )
    ]

    wrongly_negative = should_be_positive[
        should_be_positive["label"] != 1
    ]

    print("\nNegative-label audit:")
    print(
        f"  Rows inside [event-8, event-6] : "
        f"{len(should_be_positive):,}"
    )
    print(
        f"  Incorrectly labelled non-positive: "
        f"{len(wrongly_negative):,}"
    )

    # --------------------------------------------------------
    # Event timing for stays in this split
    # --------------------------------------------------------

    split_stays = set(
        features["stay_id"].unique()
    )

    split_events = events[
        events["stay_id"].isin(split_stays)
    ].copy()

    split_event_hours = split_events.loc[
        split_events["has_event"],
        "deterioration_hour"
    ]

    print("\nEvent timing for this split:")

    print(
        f"  Stays with event : "
        f"{len(split_event_hours):,}"
    )

    if len(split_event_hours) > 0:

        print(
            f"  Median event hour: "
            f"{split_event_hours.median():.2f}"
        )

        print(
            f"  Mean event hour  : "
            f"{split_event_hours.mean():.2f}"
        )

        for h in [8, 12, 24, 48, 72]:

            count = (
                split_event_hours <= h
            ).sum()

            print(
                f"  Event <= {h:>2}h : "
                f"{count:>8,}"
            )

    # --------------------------------------------------------
    # How many positive labels per stay?
    # --------------------------------------------------------

    positive_per_stay = (
        positive_rows
        .groupby("stay_id")
        .size()
    )

    print("\nPositive windows per affected stay:")

    if len(positive_per_stay) > 0:

        print(
            positive_per_stay
            .value_counts()
            .sort_index()
            .to_string()
        )

        print(
            f"\n  Affected stays : "
            f"{len(positive_per_stay):,}"
        )

        print(
            f"  Mean positives/stay : "
            f"{positive_per_stay.mean():.2f}"
        )

        print(
            f"  Max positives/stay  : "
            f"{positive_per_stay.max()}"
        )

    else:
        print("  No affected stays.")

    # --------------------------------------------------------
    # Check label hours are sorted / hourly
    # --------------------------------------------------------

    print("\nHour spacing check:")

    bad_spacing = 0

    for stay_id, group in labels.groupby(
        "stay_id",
        sort=False
    ):

        hours = np.sort(
            group["hours"].dropna().values
        )

        if len(hours) > 1:

            diffs = np.diff(hours)

            # Expected hourly grid
            bad_spacing += np.sum(
                ~np.isclose(
                    diffs,
                    1.0,
                    atol=1e-6
                )
            )

    print(
        f"  Non-1-hour gaps : "
        f"{bad_spacing:,}"
    )


print("\n" + "=" * 80)
print("AUDIT COMPLETE")
print("=" * 80)


COMPOSITE EVENT AUDIT

Total ICU stays in composite_events : 94,458
Stays with deterioration event      : 54,038
Stays without deterioration event   : 40,420

Earliest deterioration hour distribution:
count    54038.000000
mean        24.203019
std         78.263337
min          0.000000
25%          0.517778
50%          2.293472
75%          7.769653
max       2626.536667
Name: deterioration_hour, dtype: float64

Events occurring early:
  Event <=  8 hours :   40,687 (75.29%)
  Event <= 12 hours :   42,323 (78.32%)
  Event <= 24 hours :   44,931 (83.15%)
  Event <= 48 hours :   47,579 (88.05%)
  Event <= 72 hours :   49,200 (91.05%)

TRAIN SPLIT AUDIT
ERROR: Missing features file:
  data/processed/checkpoints/train_normalized.parquet

VAL SPLIT AUDIT
ERROR: Missing features file:
  data/processed/checkpoints/val_normalized.parquet

TEST SPLIT AUDIT
ERROR: Missing features file:
  data/processed/checkpoints/test_normalized.parquet

AUDIT COMPLETE
